# Package

In [2]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
import pickle

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# Model
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importation des données

In [3]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Préparation des données

## Passage en format WIDE

In [4]:
ts_raw = (
    ts_raw
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

print(ts_raw)

series_id                  UNRATE
date                             
1960-01-01 00:00:00+00:00    -0.8
1960-02-01 00:00:00+00:00    -1.1
1960-03-01 00:00:00+00:00    -0.2
1960-04-01 00:00:00+00:00     0.0
1960-05-01 00:00:00+00:00     0.0
...                           ...
2025-05-01 00:00:00+00:00     0.2
2025-06-01 00:00:00+00:00     0.0
2025-07-01 00:00:00+00:00     0.0
2025-08-01 00:00:00+00:00     0.1
2025-09-01 00:00:00+00:00     0.3

[789 rows x 1 columns]


## Construire la série y

In [5]:
# Vérifie que l’index est bien une date (sinon essaie de le convertir)
y = ts_raw.copy()

if not isinstance(y.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    y.index = pd.to_datetime(y.index, errors="coerce")

# aménager la fréquence mensuelle (début de mois)
y.index = y.index.to_period("M").to_timestamp(how="start")
y = y.sort_index().asfreq("MS").astype(float)

# 🔒 borne la date max (sans dropna)
y = y.loc[:pd.Timestamp("2025-08-01")]

# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(
    f"✅ Série prête : {y.index.min().date()} → {y.index.max().date()} "
    f"| n={len(y)} | freq={y.index.freqstr}"
)

✅ Série prête : 1960-01-01 → 2025-08-01 | n=788 | freq=MS


C:\Users\Mita\AppData\Local\Temp\ipykernel_15604\4267632309.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  y.index = y.index.to_period("M").to_timestamp(how="start")


# 2. Autoregression en choisissant automatiquement l'ordre de p

In [6]:
# ---------- Paramètres ----------
h = 12
min_train_n = 36          # ≥ 3 ans
trend = "c"               # "c" (constante) ou "n" (sans constante)
p_grid = range(1, 13)     # p ∈ {1,…,12}

cv_update_every_months = 36
cv_anchor = pd.Timestamp("1983-01-01")

# Bagging (comme les auteurs)
use_bagging = True
B_boot = 30               # n_boot ≈ 30
L_block = 12              # blocs de 12 mois (annuels)
rng = np.random.default_rng(123)  # seed bootstrap

# Conformal
use_conformal = True
alpha = 0.05
step_size = 12
pi_windows = 3           # IMPORTANT: éviter les pics (>= 24 recommandé)

In [7]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.ar_model import AutoReg

# ---------- Utils ----------
def months_since(anchor, t):
    return (t.year - anchor.year) * 12 + (t.month - anchor.month)

def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à n."""
    n = len(arr)
    L = max(2, min(int(L), n - 1))
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def rolling_mae_for_p(y_series, p, h, min_train, trend):
    """MAE rolling à l'horizon h pour un p donné (sur y_series, en respectant l'ordre temporel)."""
    rows = []
    last_t_end = y_series.index.max() - relativedelta(months=h)
    for t_end in y_series.index:
        if t_end > last_t_end:
            break
        y_tr = y_series.loc[:t_end]
        if len(y_tr) < max(min_train, p + 1):
            continue
        model = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
        fc = model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        t_fore = t_end + relativedelta(months=h)
        if t_fore in y_series.index:
            rows.append((t_fore, yhat_h, float(y_series.loc[t_fore])))
    if not rows:
        return np.inf
    tmp = pd.DataFrame(rows, columns=["date", "y_hat", "y_true"]).set_index("date")
    return float(mean_absolute_error(tmp["y_true"], tmp["y_hat"]))

def select_p_by_cv(y_tr, p_grid, h, min_train, trend):
    """Sélectionne p* minimisant le MAE(h) rolling sur l'échantillon d'entraînement courant."""
    best_p, best_score = None, np.inf
    for p in p_grid:
        score = rolling_mae_for_p(y_tr, p, h, min_train, trend)
        if score < best_score:
            best_score, best_p = score, p
    return int(best_p if best_p is not None else 1)

# ---------- AR(p) + bagging ----------
def bagged_h_forecast_ARp(y_tr, p, h, trend, B, L, rng):
    """
    Prévision à horizon h via bagging (residual moving-block bootstrap) pour AR(p).
    Retourne (yhat_mean, yhat_dist, base_pred).
    """
    base = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
    base_fc = base.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    base_pred = float(base_fc.iloc[-1])

    resid = base.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné

    preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)
        y_b = fitted + res_b
        m_b = AutoReg(
            pd.Series(y_b, index=y_tr.index[-len(y_b):]),
            lags=p, old_names=False, trend=trend
        ).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        preds.append(float(fc_b.iloc[-1]))

    return float(np.mean(preds)), np.array(preds), base_pred

def fit_predict_ar_p(y_tr, h, trend="c", p=1):
    """Fit AR(p) sur y_tr, retourne la prévision au pas h (dernier point)."""
    m = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
    fc = m.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    return float(fc.iloc[-1])

# ---------- Conformal AR(p) : calibration sur AR(p) BASE (comme ton AR1) ----------
def conformal_q_from_past_windows_pos_ARp(
    y, i_end, *,
    h=12, step_size=12, pi_windows=24,
    trend="c", p=12, alpha=0.05,
    min_train_n=36,
):
    """
    Fenêtres de calibration:
      i_cal = i_end - k*step_size
      compare y[i_cal+h] vs yhat_base(train jusqu'à i_cal)
    q = quantile(1-alpha) des |erreurs|
    """
    errs = []
    for k in range(1, pi_windows + 1):
        i_cal = i_end - k * step_size
        i_cal_fore = i_cal + h
        if i_cal < 0 or i_cal_fore >= len(y):
            continue

        y_tr_cal = y.iloc[: i_cal + 1]
        if len(y_tr_cal) < max(min_train_n, p + 2):
            continue

        try:
            yhat_cal = fit_predict_ar_p(y_tr_cal, h=h, trend=trend, p=p)  # BASE
        except Exception:
            continue

        err = abs(float(y.iloc[i_cal_fore]) - float(yhat_cal))
        if np.isfinite(err):
            errs.append(err)

    if len(errs) == 0:
        return np.nan

    # quantile conformal (finite-sample correction)
    n = len(errs)
    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(max(q_level, 0.0), 1.0)

    return float(np.quantile(errs, q_level))

In [8]:
# ---------- Boucle pseudo-OOS ----------
rows = []
p_schedule = []
last_model = None
last_fit_end = None
current_p = None

last_t_end = y.index.max() - relativedelta(months=h)

for i_end, t_end in enumerate(y.index):
    if t_end > last_t_end:
        break

    y_tr = y.loc[:t_end]
    if len(y_tr) < min_train_n:
        continue

    # Re-CV à partir de 1983-01 tous les 36 mois
    if t_end >= cv_anchor:
        m = months_since(cv_anchor, t_end)
        need_cv = (m % cv_update_every_months == 0)
    else:
        need_cv = False

    if current_p is None and not need_cv:
        current_p = 1  # valeur initiale avant la première CV

    if need_cv:
        current_p = select_p_by_cv(y_tr, p_grid, h, min_train_n, trend)
        print(f"[CV] {t_end.date()} → p* = {current_p}")
        p_schedule.append((t_end, int(current_p)))  # ✅ NEW


    # Fit de référence (utile pour meta/sauvegarde)
    arp = AutoReg(y_tr, lags=current_p, old_names=False, trend=trend).fit()
    last_model = arp
    last_fit_end = t_end

    # Prévision à h mois
    if use_bagging:
        yhat_h, yhat_dist, yhat_base = bagged_h_forecast_ARp(
            y_tr=y_tr, p=current_p, h=h, trend=trend,
            B=B_boot, L=L_block, rng=rng
        )
        # fallback bootstrap quantiles
        yhat_p05 = float(np.percentile(yhat_dist, 5))
        yhat_p95 = float(np.percentile(yhat_dist, 95))
    else:
        fc = arp.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        yhat_base = yhat_h
        yhat_dist = None
        yhat_p05 = np.nan
        yhat_p95 = np.nan

    # vérité future
    t_fore = t_end + relativedelta(months=h)
    if t_fore not in y.index:
        continue

    y_true = float(y.loc[t_fore])

    # ----- Conformal interval (AR(p) base) -----
    if use_conformal:
        q = conformal_q_from_past_windows_pos_ARp(
            y=y, i_end=i_end,
            h=h, step_size=step_size, pi_windows=pi_windows,
            trend=trend, p=current_p, alpha=alpha,
            min_train_n=min_train_n,
        )
        if np.isfinite(q):
            yhat_p05 = float(yhat_h - q)
            yhat_p95 = float(yhat_h + q)
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan

    rows.append((
        t_fore, yhat_h, y_true,
        int(current_p), yhat_p05, yhat_p95, yhat_base
    ))

[CV] 1983-01-01 → p* = 5
[CV] 1986-01-01 → p* = 4
[CV] 1989-01-01 → p* = 4
[CV] 1992-01-01 → p* = 4
[CV] 1995-01-01 → p* = 4
[CV] 1998-01-01 → p* = 4
[CV] 2001-01-01 → p* = 4
[CV] 2004-01-01 → p* = 4
[CV] 2007-01-01 → p* = 4
[CV] 2010-01-01 → p* = 4
[CV] 2013-01-01 → p* = 4
[CV] 2016-01-01 → p* = 4
[CV] 2019-01-01 → p* = 4
[CV] 2022-01-01 → p* = 4


In [11]:
# =========================================================
# Résultats + Scores par période — AR(p)
# (inclut aussi df_p_schedule si p_schedule existe)
# ✅ Fix RMSE compatible toutes versions sklearn (pas de squared=False)
# =========================================================

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------- Résultats ----------
if rows:
    df_oos_arp = (
        pd.DataFrame(
            rows,
            columns=[
                "date",
                "y_hat",
                "y_true",
                "p_used",
                "y_hat_p05",
                "y_hat_p95",
                "y_hat_base",
            ],
        )
        .assign(
            date=lambda d: pd.to_datetime(d["date"])
                            .dt.to_period("M")
                            .dt.to_timestamp(how="start")
        )
        .set_index("date")
        .sort_index()
    )
else:
    df_oos_arp = pd.DataFrame(
        columns=["y_hat", "y_true", "p_used", "y_hat_p05", "y_hat_p95", "y_hat_base"]
    )
    df_oos_arp.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_arp)}")
print(df_oos_arp.head(3))

# ---------- p* sélectionné par période (CV schedule) ----------
# p_schedule = [(t_end, p_star), ...] rempli pendant la boucle
if "p_schedule" in globals() and len(p_schedule) > 0:
    df_p_schedule = (
        pd.DataFrame(p_schedule, columns=["date", "p_star"])
        .assign(
            date=lambda d: pd.to_datetime(d["date"])
                            .dt.to_period("M")
                            .dt.to_timestamp(how="start")
        )
        .sort_values("date")
        .reset_index(drop=True)
    )
else:
    df_p_schedule = pd.DataFrame(columns=["date", "p_star"])

print(f"\n📌 CV schedule — n mises à jour p* = {len(df_p_schedule)}")
print(df_p_schedule.head(5))

# ---------- Scores par période ----------
if len(df_oos_arp):
    df_val  = df_oos_arp.loc["1983-01-01":"1989-12-31"].copy()
    df_test = df_oos_arp.loc["1990-01-01":"2025-08-31"].copy()

    if len(df_val):
        mae_val  = mean_absolute_error(df_val["y_true"], df_val["y_hat"])
        rmse_val = float(np.sqrt(mean_squared_error(df_val["y_true"], df_val["y_hat"])))
        r2_val   = r2_score(df_val["y_true"], df_val["y_hat"]) if len(df_val) > 1 else np.nan
        print(f"\n📊 Validation 83–89 — n={len(df_val)} | MAE={mae_val:.3f} | RMSE={rmse_val:.3f} | R²={r2_val:.3f}")
    else:
        mae_val, rmse_val, r2_val = np.nan, np.nan, np.nan

    if len(df_test):
        mae_test  = mean_absolute_error(df_test["y_true"], df_test["y_hat"])
        rmse_test = float(np.sqrt(mean_squared_error(df_test["y_true"], df_test["y_hat"])))
        r2_test   = r2_score(df_test["y_true"], df_test["y_hat"]) if len(df_test) > 1 else np.nan
        print(f"📊 Test 90–2025 — n={len(df_test)} | MAE={mae_test:.3f} | RMSE={rmse_test:.3f} | R²={r2_test:.3f}")
    else:
        mae_test, rmse_test, r2_test = np.nan, np.nan, np.nan

else:
    mae_val = rmse_val = r2_val = np.nan
    mae_test = rmse_test = r2_test = np.nan


✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  p_used  y_hat_p05  y_hat_p95  y_hat_base
date                                                                  
1963-12-01  0.070473     0.0       1        NaN        NaN   -0.080890
1964-01-01  0.017682    -0.1       1        NaN        NaN    0.141077
1964-02-01  0.090722    -0.5       1        NaN        NaN    0.408114

📌 CV schedule — n mises à jour p* = 14
        date  p_star
0 1983-01-01       5
1 1986-01-01       4
2 1989-01-01       4
3 1992-01-01       4
4 1995-01-01       4

📊 Validation 83–89 — n=84 | MAE=0.819 | RMSE=1.187 | R²=-0.805
📊 Test 90–2025 — n=428 | MAE=0.865 | RMSE=1.644 | R²=-0.161


In [12]:
# ==========================================
# Sauvegardes — AR(p) bagging (h=12)
# ==========================================
ARP_LAST_PKL  = "ARp_last_trained_model.pkl"
ARP_LAST_META = "ARp_last_trained_model_meta.csv"
ARP_BUNDLE    = "ARp_h12_oos_bundle.pkl"

# 1️⃣ Sauvegarde du modèle final (le dernier AR(p) entraîné)
if last_model is not None:
    try:
        joblib.dump(last_model, ARP_LAST_PKL)
        print(f"💾 Modèle AR(p) sauvegardé → {ARP_LAST_PKL}")
    except Exception:
        with open(ARP_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(p) sauvegardé (pickle) → {ARP_LAST_PKL}")

# 2️⃣ Sauvegarde du bundle complet : prévisions + paramètres + métadonnées
bundle = {
    "oos_predictions": (
        df_oos_arp.reset_index()
                  .rename(columns={"y_hat": "y_pred"})
                  .assign(date=lambda d: pd.to_datetime(d["date"]).dt.to_period("M").dt.to_timestamp(how="start"))
    ),
    "params": {
        "model": "AR(p)",
        "trend": trend,
        "horizon": h,
        "p_grid": list(p_grid),
        "min_train_n": min_train_n,
        "cv_update_every_months": cv_update_every_months,
        "cv_anchor": str(cv_anchor.date()),
        # ---- paramètres de bagging ----
        "use_bagging": bool(use_bagging),
        "B_boot": int(B_boot),
        "L_block": int(L_block)
    },
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_arp)),
        "mean_p_used": float(df_oos_arp["p_used"].mean()) if "p_used" in df_oos_arp else np.nan
    }
}

with open(ARP_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)
print(f"💾 Bundle AR(p) OOS sauvegardé → {ARP_BUNDLE}")

# 3️⃣ Sauvegarde d’un petit résumé méta au format CSV
meta_row = {
    "model": "AR(p)",
    "trend": trend,
    "horizon": h,
    "cv_anchor": str(cv_anchor.date()),
    "cv_update_months": cv_update_every_months,
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_arp)),
    "mean_p_used": float(df_oos_arp["p_used"].mean()) if "p_used" in df_oos_arp else np.nan
}

pd.DataFrame([meta_row]).to_csv(ARP_LAST_META, index=False)
print(f"💾 Méta AR(p) sauvegardée → {ARP_LAST_META}")

💾 Modèle AR(p) sauvegardé (pickle) → ARp_last_trained_model.pkl
💾 Bundle AR(p) OOS sauvegardé → ARp_h12_oos_bundle.pkl
💾 Méta AR(p) sauvegardée → ARp_last_trained_model_meta.csv


In [16]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# Vérifier la présence des métriques (mae_test / rmse_test)
# et sinon les (re)calculer proprement depuis df_oos_arp
# ---------------------------------------------------------

def _rmse(y_true, y_pred):
    # compatible toutes versions sklearn (pas de squared=False)
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# 1) Cas idéal : déjà calculées dans ta cellule "Scores par période"
mae_test_ok  = ("mae_test" in globals()) and np.isfinite(mae_test)
rmse_test_ok = ("rmse_test" in globals()) and np.isfinite(rmse_test)

if mae_test_ok and rmse_test_ok:
    print(f"✅ Métriques déjà présentes : MAE_test={mae_test:.4f} | RMSE_test={rmse_test:.4f}")

else:
    # 2) Fallback : recalcul depuis df_oos_arp (période test 1990–2025)
    df_test = df_oos_arp.loc["1990-01-01":"2025-08-31"].dropna(subset=["y_true", "y_hat"]).copy()

    if len(df_test) == 0:
        raise ValueError("❌ df_test est vide : impossible de calculer MAE/RMSE test.")

    mae_test  = float(mean_absolute_error(df_test["y_true"], df_test["y_hat"]))
    rmse_test = _rmse(df_test["y_true"], df_test["y_hat"])

    print(f"🧮 Métriques recalculées : MAE_test={mae_test:.4f} | RMSE_test={rmse_test:.4f} (n={len(df_test)})")

✅ Métriques déjà présentes : MAE_test=0.8651 | RMSE_test=1.6440


In [22]:
# =========================================================
# MLFLOW — AR(p) FINAL (ROBUSTE)
# - log métriques test
# - log bundle + p_schedule
# - log plot UNIQUEMENT si fig existe
# =========================================================

import os
import mlflow
import numpy as np
import pandas as pd

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Phase 1 : Before Machine Learning")

run_name = f"ARp_h{h}_bag{int(use_bagging)}_conf{int(use_conformal)}"

with mlflow.start_run(run_name=run_name):

    # ----------------------------
    # Params
    # ----------------------------
    mlflow.log_params(bundle["params"])

    # ----------------------------
    # Metrics (TEST)
    # ----------------------------
    mlflow.log_metrics({
        "MAE": float(mae_test),
        "RMSE": float(rmse_test),
        "n_forecasts": int(bundle["meta"]["n_forecasts"]),
        "mean_p_used": float(bundle["meta"]["mean_p_used"]),
    })

    # ----------------------------
    # Tags
    # ----------------------------
    mlflow.set_tags({
        k: ("" if v is None else str(v))
        for k, v in bundle["meta"].items()
    })

    # ----------------------------
    # Artifacts principaux
    # ----------------------------
    for f in [ARP_BUNDLE, ARP_LAST_META, ARP_LAST_PKL]:
        if os.path.exists(f):
            mlflow.log_artifact(f)

    # ----------------------------
    # Artifact : p_schedule
    # ----------------------------
    P_SCHED_CSV = "p_schedule.csv"
    bundle["p_schedule"].to_csv(P_SCHED_CSV, index=False)
    mlflow.log_artifact(P_SCHED_CSV, artifact_path="diagnostics")

    # ----------------------------
    # Artifact : plot HTML (OPTIONNEL)
    # ----------------------------
    if "fig" in globals():
        FIG_HTML = "arp_oos_plot.html"
        fig.write_html(FIG_HTML, include_plotlyjs="cdn")
        mlflow.log_artifact(FIG_HTML, artifact_path="plots")
    else:
        print("ℹ️ Plot AR(p) non loggé (fig non définie dans cette cellule)")

print(
    f"✅ MLflow AR(p) terminé | {run_name} | "
    f"MAE_test={mae_test:.4f} | RMSE_test={rmse_test:.4f}"
)

🏃 View run ARp_h12_bag1_conf1 at: http://127.0.0.1:5000/#/experiments/2/runs/c31cdeed43dd463eab9df000d690e4ba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
✅ MLflow AR(p) terminé | ARp_h12_bag1_conf1 | MAE_test=0.8651 | RMSE_test=1.6440


# Graphique

In [20]:
# =========================
# cellule 7 : Préparer df_obs + df_fcst (format utilsforecast)
# =========================
from utilsforecast.plotting import plot_series

SERIES_ID = "UNRATE"  # adapte si tu veux (ex: "UNRATE_stationary")

# df_obs : observations
df_obs = (
    df_oos_arp
    .reset_index()
    .rename(columns={"date": "ds", "y_true": "y"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "y"]]
)

# df_fcst : forecasts + intervalles
df_fcst = (
    df_oos_arp
    .reset_index()
    .rename(columns={"date": "ds", "y_hat": "AR1", "y_hat_p05": "AR1-lo-95", "y_hat_p95": "AR1-hi-95"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "AR1", "AR1-lo-95", "AR1-hi-95"]]
)

print(df_obs.head(2))
print(df_fcst.head(2))

  unique_id         ds    y
0    UNRATE 1963-12-01  0.0
1    UNRATE 1964-01-01 -0.1
  unique_id         ds       AR1  AR1-lo-95  AR1-hi-95
0    UNRATE 1963-12-01  0.070473        NaN        NaN
1    UNRATE 1964-01-01  0.017682        NaN        NaN


In [21]:
import pandas as pd
import plotly.graph_objects as go
from utilsforecast.plotting import plot_series

# =========================
# cellule 9 : Zoom + cadrage de l’axe Y
# =========================
START_ZOOM = "1990-01-01"
END_ZOOM   = "2025-08-01"

segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2019-fin"),
]

# Sécurité datetime
df_obs["ds"] = pd.to_datetime(df_obs["ds"])
df_fcst["ds"] = pd.to_datetime(df_fcst["ds"])

start_zoom_dt = pd.to_datetime(START_ZOOM)
end_zoom_dt   = pd.to_datetime(END_ZOOM)

df_obs_z = df_obs.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")
df_fcst_z = df_fcst.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")

fig = plot_series(
    df=df_obs_z,
    forecasts_df=df_fcst_z,
    level=[95],
    engine="plotly",
).update_layout(
    height=450,
    yaxis=dict(range=[-10, 12])
)

# =========================
# Renommage de la légende
# =========================
for trace in fig.data:
    name = trace.name.lower()
    if trace.name == "y":
        trace.name = "Observed"
    elif trace.name == "AR1":
        trace.name = "AR(1) forecast"
    elif "95" in name or "lo" in name or "hi" in name:
        trace.name = "Conformal Prediction"

# =========================
# Un seul bouton "Segments" (sans 1990)
# =========================
ymin, ymax = -10, 12
SEG_GROUP = "SEGMENTS"

fig.update_layout(legend=dict(groupclick="togglegroup"))

first = True
for i, (start, _, _) in enumerate(segments):

    # 👇 on ignore le premier segment (1990)
    if i == 0:
        continue

    x = pd.to_datetime(start)

    if not (start_zoom_dt <= x <= end_zoom_dt):
        continue

    fig.add_trace(
        go.Scatter(
            x=[x, x],
            y=[ymin, ymax],
            mode="lines",
            legendgroup=SEG_GROUP,
            name="Segments" if first else None,
            showlegend=first,
            visible="legendonly",
            line=dict(color="gray", width=1, dash="dash"),
            hoverinfo="skip",
        )
    )
    first = False

fig.show()

# MLops